In [1]:
import os
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

In [8]:
# ============================================================
# 1. Configuration
# ============================================================

DATA_ROOT = "./cifar10_images"
if not os.path.exists(DATA_ROOT):
    os.mkdir(DATA_ROOT)
BATCH_SIZE = 128
NUM_EPOCHS = 10
LEARNING_RATE = 1e-3
NUM_WORKERS = 2

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

Using device: cpu


In [10]:
# ============================================================
# 2. Download CIFAR-10 and save as image folders
# ============================================================

def download_and_convert_cifar10(output_root):
    """
    Download CIFAR-10 using torchvision and convert it to:

    cifar10_images/
        train/
            airplane/
            automobile/
            bird/
            ...
        val/
            airplane/
            automobile/
            bird/
            ...
    """

    output_root = Path(output_root)

    train_dir = output_root / "train"
    val_dir = output_root / "val"

    # Skip conversion if already completed
    if train_dir.exists() and val_dir.exists():
        print("Image folders already exist.")
        return

    print("Downloading CIFAR-10...")

    # Download original CIFAR-10 dataset
    cifar_train = datasets.CIFAR10(
        root="./cifar10_original",
        train=True,
        download=True
    )

    cifar_test = datasets.CIFAR10(
        root="./cifar10_original",
        train=False,
        download=True
    )

    class_names = cifar_train.classes

    # Create class directories
    for split_dir in [train_dir, val_dir]:
        for class_name in class_names:
            (split_dir / class_name).mkdir(
                parents=True,
                exist_ok=True
            )

    print("Saving training images...")

    # CIFAR-10 training set = 50,000 images
    for index, (image, label) in enumerate(cifar_train):

        class_name = class_names[label]

        image_path = (
            train_dir
            / class_name
            / f"{index:05d}.png"
        )

        image.save(image_path)

        if index % 5000 == 0:
            print(f"Saved {index}/{len(cifar_train)} training images")

    print("Saving validation images...")

    # CIFAR-10 test set = 10,000 images
    # We use this as validation data
    for index, (image, label) in enumerate(cifar_test):

        class_name = class_names[label]

        image_path = (
            val_dir
            / class_name
            / f"{index:05d}.png"
        )

        image.save(image_path)

        if index % 1000 == 0:
            print(f"Saved {index}/{len(cifar_test)} validation images")

    print("CIFAR-10 successfully converted to ImageFolder format!")


# Run download + conversion
download_and_convert_cifar10(DATA_ROOT)

100.0%


Saving training images...
Saved 0/50000 training images
Saved 5000/50000 training images
Saved 10000/50000 training images
Saved 15000/50000 training images
Saved 20000/50000 training images
Saved 25000/50000 training images
Saved 30000/50000 training images
Saved 35000/50000 training images
Saved 40000/50000 training images
Saved 45000/50000 training images
Saving validation images...
Saved 0/10000 validation images
Saved 1000/10000 validation images
Saved 2000/10000 validation images
Saved 3000/10000 validation images
Saved 4000/10000 validation images
Saved 5000/10000 validation images
Saved 6000/10000 validation images
Saved 7000/10000 validation images
Saved 8000/10000 validation images
Saved 9000/10000 validation images
CIFAR-10 successfully converted to ImageFolder format!


In [3]:
# ============================================================
# 3. Image transformations
# ============================================================

train_transform = transforms.Compose([
      transforms.Resize((224, 224)),
      transforms.RandomHorizontalFlip(), 
      transforms.ToTensor(), 
      transforms.Normalize(
          mean=(0.4914, 0.4822, 0.4465), 
          std=(0.2470, 0.2435, 0.2616) ) 
    ])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std=(0.2470, 0.2435, 0.2616)
    )
])

In [4]:
# ============================================================
# 4. Load dataset using ImageFolder
# ============================================================

train_dataset = datasets.ImageFolder(
    root=os.path.join(DATA_ROOT, "train"),
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    root=os.path.join(DATA_ROOT, "val"),
    transform=val_transform
)

print("\nClasses:")
print(train_dataset.classes)

print("\nTraining images:", len(train_dataset))
print("Validation images:", len(val_dataset))


Classes:
['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

Training images: 50000
Validation images: 10000


In [5]:
# ============================================================
# 5. DataLoaders
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

NameError: name 'BATCH_SIZE' is not defined

In [29]:
# ============================================================
# 6. Simple CNN model
# ============================================================

class SimpleCNN(nn.Module):

    def __init__(self, num_classes=10):
        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):

        x = self.features(x)
        x = self.classifier(x)

        return x


model = SimpleCNN(
    num_classes=len(train_dataset.classes)
).to(device)

model = models.alexnet(weights=None) 
# Replace final fully connected layer 
model.classifier[6] = nn.Linear( in_features=model.classifier[6].in_features, out_features=10 ) 
model = model.to(device) 
print(model)

AlexNet(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(11, 11), stride=(4, 4), padding=(2, 2))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 192, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(192, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (avgpool): AdaptiveAvgPool2d(output_size=(6, 6))
  (classifier): Sequential(
    (0): Dropout(p=0.5, inplace=False)
    (1): Linear(in_features=9216, out_features=4096, bias=True)
 

In [30]:
# ============================================================
# 7. Loss and optimizer
# ============================================================

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

In [ ]:
# ============================================================
# 8. Training and validation
# ============================================================

for epoch in range(NUM_EPOCHS):

    # ---------------------------
    # Training
    # ---------------------------

    model.train()

    train_loss = 0
    train_correct = 0
    train_total = 0
    
    print('Start Training')
    
    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

        _, predicted = outputs.max(1)

        train_total += labels.size(0)

        train_correct += (
            predicted == labels
        ).sum().item()

    train_accuracy = (
        100 * train_correct / train_total
    )

    # ---------------------------
    # Validation
    # ---------------------------

    model.eval()

    val_loss = 0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            val_loss += loss.item()

            _, predicted = outputs.max(1)

            val_total += labels.size(0)

            val_correct += (
                predicted == labels
            ).sum().item()

    val_accuracy = (
        100 * val_correct / val_total
    )

    print(
        f"Epoch [{epoch + 1}/{NUM_EPOCHS}] "
        f"Train Loss: {train_loss / len(train_loader):.4f} "
        f"Train Acc: {train_accuracy:.2f}% | "
        f"Val Loss: {val_loss / len(val_loader):.4f} "
        f"Val Acc: {val_accuracy:.2f}%"
    )

Start Training


In [ ]:

# ============================================================
# 9. Save model
# ============================================================

torch.save(
    model.state_dict(),
    "cifar10_model.pth"
)

print("\nTraining complete!")
print("Model saved as cifar10_model.pth")